# 04 - The hackathon dataset (Python)

**File:** `notebooks/04_hackathon_data_python.ipynb`

**What this does:** Loads the hackathon dataset from the local data/ folder, falling back to the bundled sample if it has not been downloaded yet.

**How to run it:** Open this file in JupyterLab, check that the kernel shown in the
top-right corner says **Python 3**, then choose *Run > Run All Cells*.

**Inputs:** files in `data/`, put there by `scripts/fetch_hackathon_data.py`

**Outputs:** a table and a short summary of it

## Where this data comes from

The hackathon dataset is the **Glider Rodeo** collection: eight glider
deployments from January 2026, each with flight and science timeseries, GPS
tracks and acoustic detections, plus raw acoustic noise files.

It lives in a **public** Google Cloud Storage bucket (`glider-rodeo-data`), so
no Google account, login or credentials are needed.

Rather than each notebook reaching out to Google Cloud, you download what you
need **once** into `data/`, and from then on every notebook just reads local
files.

Open a terminal (*File > New > Terminal*) and run this from the top of the repo
to see what is on offer:

```bash
python3 scripts/fetch_hackathon_data.py
```

Then ask for one deployment -- the whole set is 1.3 GB, so start small:

```bash
python3 scripts/fetch_hackathon_data.py sg274_20260128
```

Running it again later is safe: files you already have are skipped.

**This notebook works either way.** If nothing has been downloaded yet, it falls
back to the small `sample_stations.csv` file that ships with the repo, so every
cell still produces something.

> **If you are in Binder:** the session is erased when you close it, so anything
> you download disappears with it. A single deployment is fine to pull in Binder;
> for the full 1.3 GB use a local clone or a persistent JupyterHub.

## Where am I? Finding the repo folder

A notebook runs from the folder it lives in (`notebooks/`), **not** from the top
of the repository. So `data/sample_stations.csv` would not be found, but
`../data/sample_stations.csv` would.

Rather than writing `../` everywhere, the cell below works out where the top of
the repo is once, and builds paths from there. Every notebook here uses this
same short block.

In [ ]:
from pathlib import Path

# Path.cwd() is the folder this notebook runs in.
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

DATA = REPO / "data"

print("Repo folder:", REPO)
print("Data folder:", DATA)

## 1. What is actually in `data/`?

Two files are committed to the repo (`README.md` and `sample_stations.csv`).
Anything else you see here was downloaded by the fetch script.

In [ ]:
BUNDLED = {"README.md", "sample_stations.csv"}

all_files = sorted(p for p in DATA.rglob("*") if p.is_file())
downloaded = [p for p in all_files if p.name not in BUNDLED]

if downloaded:
    print(f"{len(downloaded)} downloaded file(s):\n")
    for path in downloaded:
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  {path.relative_to(DATA)}  ({size_mb:.2f} MB)")
else:
    print("No hackathon data downloaded yet.\n")
    print("To get it, run this from the top of the repo:")
    print("    python3 scripts/fetch_hackathon_data.py\n")
    print("Carrying on with the bundled sample file so the rest still runs.")

## 2. Choosing a file to load

Prefer a real downloaded CSV if there is one; otherwise use the sample. This is
why the notebook runs for everybody, whether or not they have the data yet.

In [ ]:
csv_files = [p for p in downloaded if p.suffix.lower() == ".csv"]

if csv_files:
    chosen = csv_files[0]
    source = "downloaded hackathon data"
else:
    chosen = DATA / "sample_stations.csv"
    source = "bundled sample (hackathon data not downloaded yet)"

print("Loading:", chosen.relative_to(REPO))
print("Source: ", source)

## 3. Loading it

In [ ]:
import pandas as pd

table = pd.read_csv(chosen)

print(f"{len(table)} rows, {len(table.columns)} columns\n")
print("Columns:", ", ".join(table.columns))

table.head()

## 4. A first look

`describe()` summarises every numeric column at once -- count, mean, min, max and
the quartiles. It is a quick way to spot missing values or impossible numbers
before trusting a dataset.

In [ ]:
table.describe()

## Try it yourself

- Change which file gets loaded by editing the index in `chosen`.
- Filter or summarise the columns that matter to your project.
- Write your results to `outputs/` -- see notebook 02 for the pattern.

## Where to go next

That is the whole starter tour: the environment (01), files and git (02), the
public AquaView catalogue (03) and the hackathon dataset (04).

From here, make your own folder as described in the
[main README](../README.md), and build something.